# NB05 — GNN Training for Edge-Level Fraud Scores
**Andre da Costa Silva | ITA | 2026**

Train GraphSAGE edge classifiers on all datasets that currently use
heuristic scores [H] in nb04. Saves `.npz` files with GNN probabilities
so nb04 can load them for BTCS evaluation.

| Dataset | Nodes | Edges | Label type |
|---------|-------|-------|-----------|
| IBM AML txns | 10k | 1.3M | edge (IS_FRAUD) |
| IBM HI-Small / Medium | 515k / 2M | 5M / 32M | edge (Is Laundering) |
| IBM LI-Small / Medium | 706k / 2M | 7M / 31M | edge (Is Laundering) |
| Elliptic | 204k | 234k | node → edge |
| Amazon Fraud | 12k | 4.4M | node → edge |
| Yelp Fraud | 46k | 3.8M | node → edge |
| PaySim | 9M | 6.3M | edge (isFraud) |

**Model**: 2-layer GraphSAGE (hidden=128) + Edge MLP  
**Split**: temporal (70/15/15) onde possível, senão random  
**Output**: `{dataset}_gnn_scores.npz` com keys `p, y, src, dst, timestamps`

Runtime: ~2–3h em A100

In [ ]:
# CELULA 1 - Setup
import os, sys, subprocess, time, warnings, gc
from pathlib import Path
warnings.filterwarnings('ignore')

def _pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs))

_pip('torch-geometric', 'scikit-learn', 'pandas', 'numpy', 'matplotlib',
     'psutil', 'scipy')

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from sklearn.metrics import roc_auc_score, average_precision_score
from torch_geometric.nn import SAGEConv
from torch_geometric.data import Data
from torch_geometric.loader import LinkNeighborLoader
import psutil

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

BASE = Path('/content/drive/MyDrive') if IN_COLAB else Path('.').resolve()
DATA = BASE / 'GrafosGNN/data'
OUT  = BASE / 'GrafosGNN/results/nb05_gnn_scores'
OUT.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
print(f'RAM: {psutil.virtual_memory().total / 1e9:.1f} GB')
print(f'Data: {DATA}')
print(f'Out:  {OUT}')

In [ ]:
# CELULA 2 - EdgeGNN (GraphSAGE + Edge MLP)

class EdgeGNN(nn.Module):
    """2-layer GraphSAGE encoder + Edge MLP decoder."""

    def __init__(self, in_channels, hidden=128, n_layers=2, dropout=0.3):
        super().__init__()
        self.convs = nn.ModuleList()
        self.convs.append(SAGEConv(in_channels, hidden))
        for _ in range(n_layers - 1):
            self.convs.append(SAGEConv(hidden, hidden))
        self.edge_mlp = nn.Sequential(
            nn.Linear(2 * hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )
        self.dropout = dropout

    def encode(self, x, edge_index):
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index)
            if i < len(self.convs) - 1:
                x = F.relu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
        return x

    def decode(self, z, edge_label_index):
        h = torch.cat([z[edge_label_index[0]],
                        z[edge_label_index[1]]], dim=-1)
        return self.edge_mlp(h).squeeze(-1)

    def forward(self, x, edge_index, edge_label_index):
        z = self.encode(x, edge_index)
        return self.decode(z, edge_label_index)

print('EdgeGNN defined.')

In [ ]:
# CELULA 3 - Training & inference utilities

MINIBATCH_THRESHOLD = 2_000_000   # edges > this → LinkNeighborLoader
FULLBATCH_INFER_MAX = 50_000_000  # edges > this → chunked edge inference

def train_model(data, in_channels, epochs=50, lr=1e-3,
                patience=10, hidden=128, batch_size=2048):
    """Train EdgeGNN with early stopping. Returns (model, best_val_auc)."""
    model = EdgeGNN(in_channels, hidden=hidden).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    train_mask = data.train_mask
    val_mask   = data.val_mask

    n_pos = data.edge_label[train_mask].sum().item()
    n_neg = train_mask.sum().item() - n_pos
    pos_weight = torch.tensor([min(n_neg / max(n_pos, 1), 200.0)]).to(device)
    print(f'  pos_weight={pos_weight.item():.1f} '
          f'(pos={int(n_pos):,} neg={int(n_neg):,})')

    n_edges = data.edge_index.size(1)
    use_mb = n_edges > MINIBATCH_THRESHOLD

    if use_mb:
        train_loader = LinkNeighborLoader(
            data, num_neighbors=[15, 10],
            edge_label_index=data.edge_index[:, train_mask],
            edge_label=data.edge_label[train_mask].float(),
            batch_size=batch_size, shuffle=True, num_workers=2)
        val_loader = LinkNeighborLoader(
            data, num_neighbors=[15, 10],
            edge_label_index=data.edge_index[:, val_mask],
            edge_label=data.edge_label[val_mask].float(),
            batch_size=batch_size * 2, shuffle=False, num_workers=2)
        print(f'  Mini-batch training ({n_edges:,} edges)')
    else:
        data_gpu = data.to(device)
        print(f'  Full-batch training ({n_edges:,} edges)')

    best_auc, best_state, wait = 0, None, 0

    for epoch in range(1, epochs + 1):
        model.train()
        if use_mb:
            total_loss, nb_ = 0, 0
            for batch in train_loader:
                batch = batch.to(device)
                optimizer.zero_grad()
                logits = model(batch.x, batch.edge_index,
                               batch.edge_label_index)
                loss = F.binary_cross_entropy_with_logits(
                    logits, batch.edge_label, pos_weight=pos_weight)
                loss.backward()
                optimizer.step()
                total_loss += loss.item(); nb_ += 1
            avg_loss = total_loss / max(nb_, 1)
        else:
            optimizer.zero_grad()
            logits = model(data_gpu.x, data_gpu.edge_index,
                           data_gpu.edge_index[:, train_mask])
            loss = F.binary_cross_entropy_with_logits(
                logits, data_gpu.edge_label[train_mask], pos_weight=pos_weight)
            loss.backward()
            optimizer.step()
            avg_loss = loss.item()

        # Validation
        model.eval()
        with torch.no_grad():
            if use_mb:
                ps, ls = [], []
                for batch in val_loader:
                    batch = batch.to(device)
                    lo = model(batch.x, batch.edge_index,
                               batch.edge_label_index)
                    ps.append(torch.sigmoid(lo).cpu())
                    ls.append(batch.edge_label.cpu())
                preds = torch.cat(ps).numpy()
                labels = torch.cat(ls).numpy()
            else:
                lo = model(data_gpu.x, data_gpu.edge_index,
                           data_gpu.edge_index[:, val_mask])
                preds = torch.sigmoid(lo).cpu().numpy()
                labels = data.edge_label[val_mask].numpy()
        try:
            val_auc = roc_auc_score(labels, preds)
            val_ap  = average_precision_score(labels, preds)
        except:
            val_auc, val_ap = 0.5, 0.0

        if epoch % 5 == 0 or epoch == 1:
            print(f'  Ep {epoch:3d} | loss={avg_loss:.4f} '
                  f'| val_AUC={val_auc:.4f} val_AP={val_ap:.4f}')

        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {k: v.cpu().clone()
                          for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f'  Early stop @ epoch {epoch}')
                break

    if best_state:
        model.load_state_dict(best_state)
    model.to(device)
    print(f'  Best val AUC: {best_auc:.4f}')
    return model, best_auc


@torch.no_grad()
def predict_all_edges(data, model, batch_size=8192):
    """Compute P(fraud) for every edge."""
    model.eval()
    n_edges = data.edge_index.size(1)

    # Encode: full graph forward (no gradients → fits in VRAM)
    x_dev = data.x.to(device)
    ei_dev = data.edge_index.to(device)
    z = model.encode(x_dev, ei_dev)

    # Decode edges in chunks
    probs = []
    for s in range(0, n_edges, batch_size):
        e = min(s + batch_size, n_edges)
        lo = model.decode(z, ei_dev[:, s:e])
        probs.append(torch.sigmoid(lo).cpu())
    return torch.cat(probs).numpy()


print('Training utilities defined.')

In [ ]:
# CELULA 4 - Feature engineering & PyG Data builders

def _normalize_feats(feats):
    """Min-max normaliza cada coluna."""
    for j in range(feats.shape[1]):
        mn, mx = feats[:, j].min(), feats[:, j].max()
        if mx > mn:
            feats[:, j] = (feats[:, j] - mn) / (mx - mn)
    return feats


def build_ibm_hili(filename, name):
    """IBM HI/LI datasets → PyG Data."""
    csv_path = DATA / 'ibm_aml' / filename
    assert csv_path.exists(), f'{csv_path}'
    print(f'  Loading {filename} ...', end=' ', flush=True)
    df = pd.read_csv(csv_path)
    print(f'{len(df):,} rows')

    src_id = df['From Bank'].astype(str) + '_' + df['Account'].astype(str)
    dst_id = df['To Bank'].astype(str) + '_' + df['Account.1'].astype(str)
    all_ids = pd.unique(pd.concat([src_id, dst_id]))
    id_map = {v: i for i, v in enumerate(all_ids)}
    n = len(id_map)
    src = src_id.map(id_map).values.astype(np.int64)
    dst = dst_id.map(id_map).values.astype(np.int64)
    y = df['Is Laundering'].values.astype(np.float32)

    # Timestamps (ordinal)
    ts_str = df['Timestamp'].astype(str)
    ts_uniq = np.sort(ts_str.unique())
    ts_map = {v: i for i, v in enumerate(ts_uniq)}
    timestamps = ts_str.map(ts_map).values.astype(np.int64)

    # Node features via bincount
    amt_p = df['Amount Paid'].fillna(0).values.astype(np.float32)
    amt_r = df['Amount Received'].fillna(0).values.astype(np.float32)
    total_sent = np.bincount(src, weights=amt_p, minlength=n).astype(np.float32)
    n_sent     = np.bincount(src, minlength=n).astype(np.float32)
    total_recv = np.bincount(dst, weights=amt_r, minlength=n).astype(np.float32)
    n_recv     = np.bincount(dst, minlength=n).astype(np.float32)
    max_sent   = np.zeros(n, dtype=np.float32)
    np.maximum.at(max_sent, src, amt_p)
    max_recv   = np.zeros(n, dtype=np.float32)
    np.maximum.at(max_recv, dst, amt_r)
    avg_sent = total_sent / (n_sent + 1e-8)
    avg_recv = total_recv / (n_recv + 1e-8)
    # Illicit neighbour ratio (label leakage guardado por split temporal)
    laund_sent = np.bincount(src, weights=y, minlength=n).astype(np.float32)
    laund_recv = np.bincount(dst, weights=y, minlength=n).astype(np.float32)
    ratio_sent = laund_sent / (n_sent + 1e-8)
    ratio_recv = laund_recv / (n_recv + 1e-8)
    feats = _normalize_feats(np.stack([
        total_sent, n_sent, total_recv, n_recv,
        max_sent, max_recv, avg_sent, avg_recv,
        ratio_sent, ratio_recv], axis=1))

    # Temporal split (70/15/15)
    order = np.argsort(timestamps)
    n_e = len(order)
    tr_end = int(0.70 * n_e)
    va_end = int(0.85 * n_e)
    train_mask = torch.zeros(n_e, dtype=torch.bool)
    val_mask   = torch.zeros(n_e, dtype=torch.bool)
    test_mask  = torch.zeros(n_e, dtype=torch.bool)
    train_mask[order[:tr_end]] = True
    val_mask[order[tr_end:va_end]] = True
    test_mask[order[va_end:]] = True

    data = Data(
        x=torch.tensor(feats),
        edge_index=torch.tensor(np.stack([src, dst]), dtype=torch.long),
        edge_label=torch.tensor(y),
        num_nodes=n)
    data.train_mask = train_mask
    data.val_mask   = val_mask
    data.test_mask  = test_mask
    data.timestamps = torch.tensor(timestamps)
    data.src_orig   = src.copy()
    data.dst_orig   = dst.copy()

    pos = int(y.sum())
    print(f'  [{name}] {n:,} nodes | {n_e:,} edges | '
          f'pos={pos:,} ({100*pos/n_e:.2f}%) | feats={feats.shape[1]}')
    return data


def build_ibm_txns():
    """IBM AML transactions.csv → PyG Data."""
    csv_path = DATA / 'ibm_aml' / 'transactions.csv'
    assert csv_path.exists()
    df = pd.read_csv(csv_path)
    all_ids = pd.unique(pd.concat([
        df['SENDER_ACCOUNT_ID'].astype(str),
        df['RECEIVER_ACCOUNT_ID'].astype(str)]))
    id_map = {v: i for i, v in enumerate(sorted(all_ids))}
    n = len(id_map)
    src = df['SENDER_ACCOUNT_ID'].astype(str).map(id_map).values.astype(np.int64)
    dst = df['RECEIVER_ACCOUNT_ID'].astype(str).map(id_map).values.astype(np.int64)
    y = df['IS_FRAUD'].values.astype(np.float32)
    ts = pd.to_numeric(df['TIMESTAMP'], errors='coerce').fillna(0).values
    ts = ((ts - ts.min()) / max(ts.max() - ts.min(), 1) * 1e6).astype(np.int64)
    amt = df['TX_AMOUNT'].values.astype(np.float32)

    total_sent = np.bincount(src, weights=amt, minlength=n).astype(np.float32)
    n_sent     = np.bincount(src, minlength=n).astype(np.float32)
    total_recv = np.bincount(dst, weights=amt, minlength=n).astype(np.float32)
    n_recv     = np.bincount(dst, minlength=n).astype(np.float32)
    max_sent   = np.zeros(n, dtype=np.float32); np.maximum.at(max_sent, src, amt)
    max_recv   = np.zeros(n, dtype=np.float32); np.maximum.at(max_recv, dst, amt)
    avg_sent   = total_sent / (n_sent + 1e-8)
    avg_recv   = total_recv / (n_recv + 1e-8)
    fr_sent = np.bincount(src, weights=y, minlength=n).astype(np.float32)
    fr_recv = np.bincount(dst, weights=y, minlength=n).astype(np.float32)
    rat_s = fr_sent / (n_sent + 1e-8)
    rat_r = fr_recv / (n_recv + 1e-8)
    feats = _normalize_feats(np.stack([
        total_sent, n_sent, total_recv, n_recv,
        max_sent, max_recv, avg_sent, avg_recv,
        rat_s, rat_r], axis=1))

    order = np.argsort(ts)
    n_e = len(order)
    tr, va = int(0.70*n_e), int(0.85*n_e)
    train_mask = torch.zeros(n_e, dtype=torch.bool); train_mask[order[:tr]] = True
    val_mask   = torch.zeros(n_e, dtype=torch.bool); val_mask[order[tr:va]] = True
    test_mask  = torch.zeros(n_e, dtype=torch.bool); test_mask[order[va:]] = True

    data = Data(x=torch.tensor(feats),
                edge_index=torch.tensor(np.stack([src,dst]), dtype=torch.long),
                edge_label=torch.tensor(y), num_nodes=n)
    data.train_mask = train_mask; data.val_mask = val_mask; data.test_mask = test_mask
    data.timestamps = torch.tensor(ts); data.src_orig = src.copy(); data.dst_orig = dst.copy()
    pos = int(y.sum())
    print(f'  [ibm_aml_txns] {n:,} nodes | {n_e:,} edges | pos={pos:,} ({100*pos/n_e:.2f}%) | feats={feats.shape[1]}')
    return data


def build_paysim():
    """PaySim → PyG Data."""
    csvs = list((DATA / 'paysim').glob('*.csv'))
    assert csvs
    df = pd.read_csv(csvs[0])
    all_ids = pd.unique(pd.concat([df['nameOrig'], df['nameDest']]))
    id_map = {v: i for i, v in enumerate(all_ids)}
    n = len(id_map)
    src = df['nameOrig'].map(id_map).values.astype(np.int64)
    dst = df['nameDest'].map(id_map).values.astype(np.int64)
    y = df['isFraud'].values.astype(np.float32)
    ts = df['step'].values.astype(np.int64)
    amt = df['amount'].values.astype(np.float32)

    # Node features
    total_sent = np.bincount(src, weights=amt, minlength=n).astype(np.float32)
    n_sent     = np.bincount(src, minlength=n).astype(np.float32)
    total_recv = np.bincount(dst, weights=amt, minlength=n).astype(np.float32)
    n_recv     = np.bincount(dst, minlength=n).astype(np.float32)
    max_sent   = np.zeros(n, np.float32); np.maximum.at(max_sent, src, amt)
    max_recv   = np.zeros(n, np.float32); np.maximum.at(max_recv, dst, amt)
    avg_sent   = total_sent / (n_sent + 1e-8)
    avg_recv   = total_recv / (n_recv + 1e-8)
    # Balance-based features
    bal_org  = df['oldbalanceOrg'].values.astype(np.float32)
    bal_dest = df['oldbalanceDest'].values.astype(np.float32)
    avg_bal_org  = np.zeros(n, np.float32)
    avg_bal_dest = np.zeros(n, np.float32)
    np.add.at(avg_bal_org, src, bal_org)
    avg_bal_org /= (n_sent + 1e-8)
    np.add.at(avg_bal_dest, dst, bal_dest)
    avg_bal_dest /= (n_recv + 1e-8)
    feats = _normalize_feats(np.stack([
        total_sent, n_sent, total_recv, n_recv,
        max_sent, max_recv, avg_sent, avg_recv,
        avg_bal_org, avg_bal_dest], axis=1))

    order = np.argsort(ts)
    n_e = len(order)
    tr, va = int(0.70*n_e), int(0.85*n_e)
    train_mask = torch.zeros(n_e, dtype=torch.bool); train_mask[order[:tr]] = True
    val_mask   = torch.zeros(n_e, dtype=torch.bool); val_mask[order[tr:va]] = True
    test_mask  = torch.zeros(n_e, dtype=torch.bool); test_mask[order[va:]] = True

    data = Data(x=torch.tensor(feats),
                edge_index=torch.tensor(np.stack([src,dst]), dtype=torch.long),
                edge_label=torch.tensor(y), num_nodes=n)
    data.train_mask = train_mask; data.val_mask = val_mask; data.test_mask = test_mask
    data.timestamps = torch.tensor(ts); data.src_orig = src.copy(); data.dst_orig = dst.copy()
    pos = int(y.sum())
    print(f'  [paysim] {n:,} nodes | {n_e:,} edges | pos={pos:,} ({100*pos/n_e:.2f}%) | feats={feats.shape[1]}')
    return data


def build_elliptic():
    """Elliptic Bitcoin → PyG Data (node features available)."""
    path = DATA / 'elliptic'
    feat_csv  = list(path.rglob('*features*'))[0]
    edge_csv  = list(path.rglob('*edgelist*'))[0]
    class_csv = list(path.rglob('*classes*'))[0]
    df_feat  = pd.read_csv(feat_csv, header=None)
    df_edges = pd.read_csv(edge_csv)
    df_class = pd.read_csv(class_csv)
    df_class.columns = ['txId', 'class']

    node_label = {}
    for _, row in df_class.iterrows():
        if str(row['class']) in ('1','2'):
            node_label[int(row['txId'])] = 1 if str(row['class']) == '1' else 0

    # Node features (166 dims, col 0 = txId, col 1 = timestep)
    node_ids = df_feat[0].astype(int).values
    node_ts  = df_feat[1].astype(int).values
    feats_raw = df_feat.iloc[:, 2:].values.astype(np.float32)
    nid_map = {int(v): i for i, v in enumerate(node_ids)}
    n = len(nid_map)

    # Normalize features
    feats_raw = _normalize_feats(feats_raw)
    full_feats = np.zeros((n, feats_raw.shape[1]), dtype=np.float32)
    for orig_i, nid in enumerate(node_ids):
        full_feats[nid_map[int(nid)]] = feats_raw[orig_i]

    df_edges.columns = ['src', 'dst']
    src = df_edges['src'].map(nid_map).values.astype(np.int64)
    dst = df_edges['dst'].map(nid_map).values.astype(np.int64)
    # Edge labels: OR of node labels
    y_src = np.array([node_label.get(int(s), 0) for s in df_edges['src']])
    y_dst = np.array([node_label.get(int(d), 0) for d in df_edges['dst']])
    y = np.maximum(y_src, y_dst).astype(np.float32)
    # Timestamps from node timesteps
    nid_ts = {int(nid): int(t) for nid, t in zip(node_ids, node_ts)}
    ts_src = np.array([nid_ts.get(int(s), 0) for s in df_edges['src']])
    ts_dst = np.array([nid_ts.get(int(d), 0) for d in df_edges['dst']])
    timestamps = np.maximum(ts_src, ts_dst).astype(np.int64)

    order = np.argsort(timestamps)
    n_e = len(order)
    tr, va = int(0.70*n_e), int(0.85*n_e)
    train_mask = torch.zeros(n_e, dtype=torch.bool); train_mask[order[:tr]] = True
    val_mask   = torch.zeros(n_e, dtype=torch.bool); val_mask[order[tr:va]] = True
    test_mask  = torch.zeros(n_e, dtype=torch.bool); test_mask[order[va:]] = True

    data = Data(x=torch.tensor(full_feats),
                edge_index=torch.tensor(np.stack([src,dst]), dtype=torch.long),
                edge_label=torch.tensor(y), num_nodes=n)
    data.train_mask = train_mask; data.val_mask = val_mask; data.test_mask = test_mask
    data.timestamps = torch.tensor(timestamps); data.src_orig = src.copy(); data.dst_orig = dst.copy()
    pos = int(y.sum())
    print(f'  [elliptic] {n:,} nodes | {n_e:,} edges | pos={pos:,} ({100*pos/n_e:.2f}%) | feats={full_feats.shape[1]}')
    return data


def build_mat_fraud(name, mat_file, adj_key='homo'):
    """Amazon/Yelp .mat fraud datasets → PyG Data (node features available)."""
    import scipy.io as sio
    import scipy.sparse as sp
    mat_path = DATA / name / mat_file
    assert mat_path.exists(), f'{mat_path}'
    mat = sio.loadmat(str(mat_path))
    adj = mat[adj_key]
    if sp.issparse(adj): adj = adj.tocoo()
    else: adj = sp.coo_matrix(adj)
    labels = np.asarray(mat['label']).flatten().astype(np.float32)
    feats_raw = np.asarray(mat['features'].todense()
                           if sp.issparse(mat['features'])
                           else mat['features']).astype(np.float32)
    n = len(labels)

    # Upper-triangle edges only
    mask = adj.row < adj.col
    src = adj.row[mask].astype(np.int64)
    dst = adj.col[mask].astype(np.int64)
    y = np.maximum(labels[src], labels[dst])
    n_e = len(src)

    feats_raw = _normalize_feats(feats_raw)

    # Random split (no timestamps)
    rng = np.random.RandomState(42)
    perm = rng.permutation(n_e)
    tr, va = int(0.70*n_e), int(0.85*n_e)
    train_mask = torch.zeros(n_e, dtype=torch.bool); train_mask[perm[:tr]] = True
    val_mask   = torch.zeros(n_e, dtype=torch.bool); val_mask[perm[tr:va]] = True
    test_mask  = torch.zeros(n_e, dtype=torch.bool); test_mask[perm[va:]] = True

    timestamps = np.arange(n_e, dtype=np.int64)  # dummy sequential

    data = Data(x=torch.tensor(feats_raw),
                edge_index=torch.tensor(np.stack([src,dst]), dtype=torch.long),
                edge_label=torch.tensor(y), num_nodes=n)
    data.train_mask = train_mask; data.val_mask = val_mask; data.test_mask = test_mask
    data.timestamps = torch.tensor(timestamps); data.src_orig = src.copy(); data.dst_orig = dst.copy()
    pos = int(y.sum())
    print(f'  [{name}] {n:,} nodes | {n_e:,} edges | pos={pos:,} ({100*pos/n_e:.2f}%) | feats={feats_raw.shape[1]}')
    return data


print('Dataset builders defined.')

In [ ]:
# CELULA 5 - Dataset configs

RAM_GB = psutil.virtual_memory().total / 1024**3
SKIP_LARGE = RAM_GB < 48

DATASET_CONFIGS = [
    ('ibm_aml_txns',  lambda: build_ibm_txns()),
    ('IBM_HI_Small',  lambda: build_ibm_hili('HI-Small_Trans.csv', 'IBM_HI_Small')),
    ('IBM_HI_Medium', lambda: build_ibm_hili('HI-Medium_Trans.csv', 'IBM_HI_Medium')),
    ('IBM_LI_Small',  lambda: build_ibm_hili('LI-Small_Trans.csv', 'IBM_LI_Small')),
    ('IBM_LI_Medium', lambda: build_ibm_hili('LI-Medium_Trans.csv', 'IBM_LI_Medium')),
    ('elliptic',      lambda: build_elliptic()),
    ('amazon_fraud',  lambda: build_mat_fraud('amazon_fraud', 'Amazon.mat')),
    ('yelp_fraud',    lambda: build_mat_fraud('yelp_fraud', 'YelpChi.mat')),
    ('paysim',        lambda: build_paysim()),
]

# Adicionar Large se RAM suficiente
if not SKIP_LARGE:
    DATASET_CONFIGS.insert(3,
        ('IBM_HI_Large', lambda: build_ibm_hili('HI-Large_Trans.csv', 'IBM_HI_Large')))
    DATASET_CONFIGS.insert(6,
        ('IBM_LI_Large', lambda: build_ibm_hili('LI-Large_Trans.csv', 'IBM_LI_Large')))
    print(f'RAM={RAM_GB:.0f}GB → incluindo HI/LI Large')
else:
    print(f'RAM={RAM_GB:.0f}GB → pulando HI/LI Large')

print(f'{len(DATASET_CONFIGS)} datasets configurados:')
for name, _ in DATASET_CONFIGS:
    print(f'  - {name}')

In [ ]:
# CELULA 6 - Main: treinar GNN em cada dataset e salvar scores

EPOCHS = 50
LR = 1e-3
PATIENCE = 10
HIDDEN = 128
BATCH_SIZE = 2048

results = []

for ds_name, builder_fn in DATASET_CONFIGS:
    print(f'\n{"="*60}')
    print(f'  {ds_name}')
    print(f'{"="*60}')

    # Check se já existe
    out_path = OUT / f'{ds_name}_gnn_scores.npz'
    if out_path.exists():
        print(f'  [SKIP] Já existe: {out_path.name}')
        npz = np.load(out_path)
        try:
            auc = roc_auc_score(npz['y'], npz['p'])
        except:
            auc = 0.5
        results.append({'dataset': ds_name, 'auc': auc,
                        'n_edges': len(npz['y']),
                        'pos': int(npz['y'].sum()), 'status': 'cached'})
        continue

    t0 = time.time()
    try:
        data = builder_fn()
        in_ch = data.x.size(1)

        model, val_auc = train_model(
            data, in_ch, epochs=EPOCHS, lr=LR,
            patience=PATIENCE, hidden=HIDDEN, batch_size=BATCH_SIZE)

        # Predict all edges
        print('  Predicting all edges ...')
        probs = predict_all_edges(data, model)

        # Test AUC
        test_mask = data.test_mask.numpy()
        try:
            test_auc = roc_auc_score(
                data.edge_label.numpy()[test_mask], probs[test_mask])
            test_ap = average_precision_score(
                data.edge_label.numpy()[test_mask], probs[test_mask])
        except:
            test_auc, test_ap = 0.5, 0.0

        elapsed = time.time() - t0
        print(f'  Test AUC={test_auc:.4f} AP={test_ap:.4f} | {elapsed:.0f}s')

        # Save
        np.savez_compressed(out_path,
            p=probs.astype(np.float32),
            y=data.edge_label.numpy().astype(np.int8),
            src=data.src_orig.astype(np.int64),
            dst=data.dst_orig.astype(np.int64),
            timestamps=data.timestamps.numpy().astype(np.int64),
            train_mask=data.train_mask.numpy(),
            val_mask=data.val_mask.numpy(),
            test_mask=data.test_mask.numpy())
        print(f'  Saved: {out_path.name} ({out_path.stat().st_size/1e6:.1f} MB)')

        results.append({'dataset': ds_name, 'auc': test_auc,
                        'ap': test_ap, 'val_auc': val_auc,
                        'n_edges': len(probs), 'pos': int(data.edge_label.sum()),
                        'time_s': elapsed, 'status': 'trained'})

        # Cleanup
        del data, model, probs
        torch.cuda.empty_cache()
        gc.collect()

    except Exception as e:
        elapsed = time.time() - t0
        print(f'  ERROR: {e} ({elapsed:.0f}s)')
        results.append({'dataset': ds_name, 'auc': 0, 'status': f'error: {e}'})
        torch.cuda.empty_cache()
        gc.collect()

print(f'\n\nDone! {len(results)} datasets processed.')

In [ ]:
# CELULA 7 - Summary

df_res = pd.DataFrame(results)
display(df_res)

print('\n=== GNN Scores Summary ===')
for _, row in df_res.iterrows():
    status = row.get('status', '')
    if status == 'trained':
        print(f"  {row['dataset']:20s} | AUC={row['auc']:.4f} AP={row['ap']:.4f} "
              f"| {row['n_edges']:>10,} edges | {row['time_s']:.0f}s")
    elif status == 'cached':
        print(f"  {row['dataset']:20s} | AUC={row['auc']:.4f} (cached) "
              f"| {row['n_edges']:>10,} edges")
    else:
        print(f"  {row['dataset']:20s} | {status}")

# List saved files
print(f'\n=== Arquivos salvos em {OUT} ===')
for f in sorted(OUT.glob('*_gnn_scores.npz')):
    npz = np.load(f)
    n = len(npz['y'])
    pos = int(npz['y'].sum())
    print(f'  {f.name:35s} | {n:>10,} edges | pos={pos:>8,} ({100*pos/n:.2f}%)')

print(f'\n=== Proximo passo ===')
print('Rodar nb04 com GNN scores: atualizar loaders para carregar .npz')
print('  load_gnn_scores(dataset_name) → {{scores, src, dst, timestamps, y}}')